In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import html
import pandas as pd
from pathlib import Path
import os
from src.utils.config import find_project_root, load_config

In [2]:
# =============================================================================
# SECTION 0: Configuration
# =============================================================================

# Determine root directory of project and load configuration file
project_root = find_project_root()
config = load_config()

# Get current working directory 
pwd = os.getcwd()

# Specify path to loan-level address data
address_dir = os.path.join(pwd,'geocoding_input')

# Path to the input data file (pre-filtered to disagreements only).
# Supported formats: .parquet or .csv
INPUT_DATA_PATH = os.path.join(address_dir,'consistent_parsed_addresses_random_sample.parquet')

# Folder where review_results.parquet will be saved.
# Created automatically if it does not exist.
OUTPUT_FOLDER = os.path.join(pwd,'manual_review/random_sample_validation')

# Number of cases to review per session.
BATCH_SIZE = 10

In [3]:
# =============================================================================
# SECTION 1: Load input data
# =============================================================================

input_path = Path(INPUT_DATA_PATH)

if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path.resolve()}")

if input_path.suffix == ".parquet":
    df = pd.read_parquet(input_path)
elif input_path.suffix == ".csv":
    df = pd.read_csv(input_path)
else:
    raise ValueError(
        f"Unsupported format: '{input_path.suffix}'. Please provide a .parquet or .csv file."
    )

print(f"Loaded {len(df)} row(s) from: {input_path.resolve()}")

# =============================================================================
# SECTION 2: Load existing results and identify already-processed cases
# =============================================================================

output_folder = Path(OUTPUT_FOLDER)
output_folder.mkdir(parents=True, exist_ok=True)
output_path = output_folder / "accuracy_review_results.parquet"

if output_path.exists():
    existing_results_df = pd.read_parquet(output_path)
    processed_ids = set(existing_results_df["masterloanidtrepp"].tolist())
    print(f"Found existing results: {len(existing_results_df)} case(s) already reviewed.")
else:
    existing_results_df = pd.DataFrame()
    processed_ids = set()
    print("No existing results found. Starting fresh.")

# =============================================================================
# SECTION 3: Build the unprocessed case list and apply batch limit
# =============================================================================
# Unlike the disagreement review notebook, each loan here has exactly one row.
# We iterate over rows directly rather than grouping by loan ID.

all_unprocessed = []
for _, row in df.iterrows():
    if row["masterloanidtrepp"] in processed_ids:
        continue
    all_unprocessed.append(row)

case_list = all_unprocessed[:BATCH_SIZE]

# Captured by _advance() below to report how many cases remain after this batch.
remaining_after_batch = len(all_unprocessed) - len(case_list)

if not case_list:
    print("\n✅ All cases have already been reviewed. Nothing to do.")
else:
    print(
        f"\n{len(all_unprocessed)} unprocessed case(s) found.\n"
        f"This session: reviewing {len(case_list)} case(s). "
        f"{remaining_after_batch} case(s) will remain after this batch."
    )

# =============================================================================
# SECTION 4: Schema validation
# =============================================================================

_VALID_ADDRESS_TYPES = {"exact", "range", "approximate"}


def validate_address_components(parsed):
    """
    Validate a parsed address_components value against the required schema.
    Returns a list of error strings. An empty list means the input is valid.

    Schema rules per element:
      address_type           : "exact" | "range" | "approximate"
      first_building_number  : integer when address_type is "exact"/"range";
                               null    when address_type is "approximate"
      last_building_number   : same rules as first_building_number
      street                 : non-empty string
    """
    errors = []

    if not isinstance(parsed, list):
        return ["Top-level value must be a JSON array, e.g. [ { ... } ]."]
    if len(parsed) == 0:
        return ["The array must contain at least one element."]

    for i, element in enumerate(parsed):
        label = f"Element {i + 1}"

        if not isinstance(element, dict):
            errors.append(
                f"{label}: expected a JSON object, got {type(element).__name__}."
            )
            continue  # can't check individual fields without a dict

        required_keys   = {"address_type", "first_building_number",
                           "last_building_number", "street"}
        present_keys    = set(element.keys())
        missing_keys    = required_keys - present_keys
        unexpected_keys = present_keys - required_keys

        if missing_keys:
            errors.append(
                f"{label}: missing required key(s): "
                f"{', '.join(repr(k) for k in sorted(missing_keys))}."
            )
        if unexpected_keys:
            errors.append(
                f"{label}: unexpected key(s): "
                f"{', '.join(repr(k) for k in sorted(unexpected_keys))}."
            )

        addr_type = element.get("address_type")
        if "address_type" in element:
            if addr_type not in _VALID_ADDRESS_TYPES:
                errors.append(
                    f"{label}: 'address_type' must be one of "
                    f"{sorted(_VALID_ADDRESS_TYPES)}, got {repr(addr_type)}."
                )

        for field in ("first_building_number", "last_building_number"):
            if field not in element:
                continue  # missing key already reported above

            val = element[field]

            if addr_type == "approximate":
                if val is not None:
                    errors.append(
                        f"{label}: '{field}' must be null when "
                        f"address_type is 'approximate' (got {repr(val)})."
                    )
            else:
                if val is None:
                    errors.append(
                        f"{label}: '{field}' must be an integer when "
                        f"address_type is '{addr_type}' (null is not allowed)."
                    )
                elif isinstance(val, bool) or not isinstance(val, int):
                    # bool is excluded explicitly because in Python bool is a
                    # subclass of int, so isinstance(True, int) is True.
                    errors.append(
                        f"{label}: '{field}' must be an integer, "
                        f"got {type(val).__name__} ({repr(val)})."
                    )

        if "street" in element:
            street = element["street"]
            if not isinstance(street, str) or not street.strip():
                errors.append(
                    f"{label}: 'street' must be a non-empty string "
                    f"(got {repr(street)})."
                )

    return errors


# =============================================================================
# SECTION 5: Equivalence check
# =============================================================================

def are_equivalent(human_parsed, llm_parsed):
    """
    Return True if the human-parsed and LLM-parsed address_components are identical.

    Comparison properties (per the project specification):
      - Case-sensitive and whitespace-sensitive for string fields
      - Integer equality for building numbers
      - None == None for approximate addresses
      - Order-sensitive across array elements

    Python's built-in == operator performs a deep element-wise comparison on
    nested lists of dicts, satisfying all of the above properties automatically.
    """
    return human_parsed == llm_parsed


# =============================================================================
# SECTION 6: Session state
# =============================================================================

# Accumulates results recorded during this session.
session_results = []

# Mutable dict used to share state between event handlers.
# A dict is used (rather than plain variables) because Python closures
# do not allow rebinding of outer-scope names inside nested functions.
state = {
    "current_index": 0,
    # Stores the most recently validated human parse for the current case.
    # Updated both in Stage 1 (on_stage1_submit) and Stage 2 (on_stage2_submit).
    "human_parsed":  None,
    # Tracks which stage is currently active; useful for debugging.
    "stage":         "human_input",   # "human_input" or "comparison"
}

# =============================================================================
# SECTION 7: Create widgets
# =============================================================================

# --- Shared widgets (visible in both stages) ---

progress_label = widgets.HTML(value="")

# Displays loan ID and address string prominently so it remains visible
# across both stages without duplication.
info_html = widgets.HTML(value="")

# ---------------------------------------------------------------------------
# Stage 1: Human input
# The address string is shown; the LLM parse is deliberately withheld.
# ---------------------------------------------------------------------------

# Template string used to pre-populate the Stage 1 textarea.
# json.dumps with indent=2 produces the exact multi-line format requested.
_STAGE1_TEMPLATE = json.dumps(
    [
        {
            "address_type": "exact",
            "first_building_number": 27,
            "last_building_number": 27,
            "street": "Rogerson Drive",
        }
    ],
    indent=2,
)

# A compact schema reference collapsed into an accordion to minimise
# the distance between the address string and the text input box.
_schema_detail_html = widgets.HTML(
    value=(
        '<div style="font-size:12px; padding:4px 2px;">'
        "<b>Required keys per element:</b>"
        '<ul style="margin:4px 0; padding-left:18px;">'
        "<li><code>address_type</code>: "
        '<code>"exact"</code> (single number), '
        '<code>"range"</code> (hyphenated range, e.g. 442–446), '
        '<code>"approximate"</code> (no numeric part, e.g. "One Grumman Road")</li>'
        "<li><code>first_building_number</code>: integer, or "
        "<code>null</code> for approximate addresses</li>"
        "<li><code>last_building_number</code>: same rule as above; "
        "set equal to <code>first_building_number</code> for exact addresses</li>"
        "<li><code>street</code>: full name including any directional "
        '(e.g. "South Glenstone Avenue", "Route 35 South")</li>'
        "</ul>"
        "</div>"
    )
)

_schema_accordion = widgets.Accordion(
    children=[_schema_detail_html],
    layout=widgets.Layout(margin="0 0 6px 0"),
)
_schema_accordion.set_title(0, "Schema reference (click to expand)")
_schema_accordion.selected_index = None   # collapsed by default

stage1_instructions = widgets.VBox(
    [
        widgets.HTML(
            value=(
                '<div style="font-size:12px; color:#555; margin:4px 0 2px 0;">'
                "Parse the address string into a JSON array. "
                "Edit the template below — "
                "validation runs on submission."
                "</div>"
            )
        ),
        _schema_accordion,
    ],
    layout=widgets.Layout(margin="0 0 4px 0"),
)

# Pre-populated with the template so the reviewer can edit in place
# rather than typing from scratch.
stage1_textarea = widgets.Textarea(
    value=_STAGE1_TEMPLATE,
    layout=widgets.Layout(width="100%", height="180px"),
)

stage1_feedback = widgets.Output()

stage1_submit = widgets.Button(
    description="Submit Parse →",
    button_style="primary",
    icon="check",
    layout=widgets.Layout(width="165px"),
)

stage1_container = widgets.VBox(
    [stage1_instructions, stage1_textarea, stage1_feedback, stage1_submit],
    layout=widgets.Layout(margin="4px 0"),
)

# ---------------------------------------------------------------------------
# Stage 2: Comparison view
# Shown only when the human parse does not match the LLM parse.
# Displays both parses side by side with options to revise or flag the LLM.
# ---------------------------------------------------------------------------

# Prominent notice explaining why Stage 2 has appeared.
stage2_notice = widgets.HTML(value="")

# Updated by load_case() / show_stage2() / on_stage2_submit()
# to reflect the most recent human parse.
stage2_human_html = widgets.HTML(value="")

# Set once by show_stage2() and does not change for the current case.
stage2_llm_html = widgets.HTML(value="")

stage2_radio = widgets.RadioButtons(
    options=["Revise my parse", "Mark LLM as incorrect"],
    value="Revise my parse",
    description="Action:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="auto"),
)

stage2_revise_instructions = widgets.HTML(
    value=(
        '<span style="color:#555; font-size:12px;">'
        "Your current parse has been pre-loaded below. "
        "Edit as needed and click Submit."
        "</span>"
    )
)

stage2_revise_textarea = widgets.Textarea(
    layout=widgets.Layout(width="100%", height="180px"),
)

# Group instructions and textarea so both can be shown/hidden with one toggle.
stage2_revise_section = widgets.VBox(
    [stage2_revise_instructions, stage2_revise_textarea],
    layout=widgets.Layout(margin="4px 0 0 0"),
)

stage2_feedback = widgets.Output()

stage2_submit = widgets.Button(
    description="Submit →",
    button_style="primary",
    icon="check",
    layout=widgets.Layout(width="150px"),
)

# Stage 2 is hidden on startup; show_stage2() reveals it.
stage2_container = widgets.VBox(
    [
        stage2_notice,
        stage2_human_html,
        stage2_llm_html,
        widgets.HTML('<hr style="margin:4px 0;">'),
        stage2_radio,
        stage2_revise_section,
        stage2_feedback,
        stage2_submit,
    ],
    layout=widgets.Layout(display="none", margin="4px 0"),
)

completion_output = widgets.Output()


# =============================================================================
# SECTION 8: Helper functions
# =============================================================================

def parse_components(value):
    """
    Convert address_components to a Python object.
    Handles both JSON-encoded strings (common when loading from CSV/parquet)
    and already-parsed Python objects (list, dict).
    """
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value    # return raw string rather than crashing
    return value


def format_components_html(parsed, label):
    """
    Render an already-parsed address_components object as a styled HTML block.
    Accepts a Python object (list/dict) rather than a raw string.
    html.escape() prevents special characters from breaking the HTML structure.
    """
    try:
        pretty = json.dumps(parsed, indent=2)
    except TypeError:
        pretty = str(parsed)

    escaped = html.escape(pretty)

    return (
        f'<div style="margin-bottom:8px;">'
        f"<b>{html.escape(label)}</b>"
        f'<pre style="background:#f8f8f8; border:1px solid #ddd; '
        f"border-radius:4px; padding:8px; font-size:12px; "
        f'white-space:pre-wrap; word-break:break-word;">'
        f"{escaped}</pre></div>"
    )


def load_case(index):
    """
    Populate shared widgets with data for the case at the given index,
    reset all controls to their default state, and display Stage 1.
    Called at startup and after each case is submitted.
    """
    row         = case_list[index]
    loan_id     = row["masterloanidtrepp"]
    address_str = row["address_string"]

    progress_label.value = (
        f"<b>Case {index + 1} of {len(case_list)}</b> "
        f'<span style="color:#888; font-weight:normal;">'
        f"({len(session_results)} submitted this session)</span>"
    )

    # Make the address string large and prominent — it is the key input
    # the reviewer needs to see clearly throughout both stages.
    info_html.value = (
        f'<div style="margin:8px 0;">'
        f"<b>Loan ID:</b> {html.escape(str(loan_id))}<br>"
        f"<b>Address to parse:</b> "
        f'<span style="font-family:monospace; font-size:15px; '
        f'font-weight:bold; color:#1a5276;">'
        f"{html.escape(str(address_str))}</span>"
        f"</div>"
    )

    # Reset mutable state for the new case
    state["human_parsed"] = None
    state["stage"]        = "human_input"

    # Reset Stage 1 controls; restore the template so the reviewer has a
    # clean starting point for each new case.
    stage1_textarea.value    = _STAGE1_TEMPLATE
    stage1_textarea.disabled = False
    stage1_submit.disabled   = False
    stage1_feedback.clear_output()

    # Pre-reset Stage 2 controls so they are in the correct state when
    # show_stage2() reveals the container for this case.
    stage2_radio.value                   = "Revise my parse"
    stage2_revise_textarea.value         = ""
    stage2_revise_textarea.disabled      = False
    # Pre-set revise section as visible; paired with the default radio value
    # "Revise my parse" so Stage 2 appears in a consistent initial state.
    stage2_revise_section.layout.display = ""
    stage2_submit.disabled               = False
    stage2_feedback.clear_output()

    # Show Stage 1, hide Stage 2
    stage1_container.layout.display = ""
    stage2_container.layout.display = "none"


def show_stage2():
    """
    Switch from Stage 1 to Stage 2 (comparison view).
    Called when a valid human parse does not match the LLM output.
    Populates the comparison widgets without overwriting any widget values
    that the reviewer has not yet interacted with.
    """
    row        = case_list[state["current_index"]]
    llm_parsed = parse_components(row["address_components"])

    stage2_notice.value = (
        '<div style="background:#fdedec; border:1px solid #e74c3c; '
        'border-radius:4px; padding:10px; margin:8px 0; color:#922b21;">'
        "<b>⚠️  Your parse does not match the LLM output.</b><br>"
        "Review both parses below. You can revise your parse (e.g. to correct a typo), "
        "or mark the LLM output as incorrect if you believe your parse is right."
        "</div>"
    )

    stage2_human_html.value = format_components_html(
        state["human_parsed"], "Your parse:"
    )
    stage2_llm_html.value = format_components_html(
        llm_parsed, "LLM parse:"
    )

    # Pre-populate the revise textarea with the human's current (validated) parse
    # so they can make targeted edits rather than re-typing from scratch.
    try:
        stage2_revise_textarea.value = json.dumps(state["human_parsed"], indent=2)
    except TypeError:
        stage2_revise_textarea.value = ""

    stage2_radio.value                   = "Revise my parse"
    stage2_revise_section.layout.display = ""    # visible for the default radio value
    stage2_feedback.clear_output()

    # Hide Stage 1, reveal Stage 2
    stage1_container.layout.display = "none"
    stage2_container.layout.display = ""

    state["stage"] = "comparison"


def _record_result(llm_correct):
    """
    Append the result for the current case to session_results.
    Both address_components columns are stored as canonical JSON strings
    for consistent parquet serialisation.
    """
    row        = case_list[state["current_index"]]
    llm_parsed = parse_components(row["address_components"])

    session_results.append({
        "masterloanidtrepp":        row["masterloanidtrepp"],
        "address_string":           row["address_string"],
        "human_address_components": json.dumps(state["human_parsed"]),
        "llm_address_components":   json.dumps(llm_parsed),
        "llm_correct":              llm_correct,
    })


def save_results():
    """
    Merge session results with any previously saved results and write
    the combined dataset to output_path as a parquet file.
    Returns the total number of rows saved.
    """
    session_df = pd.DataFrame(session_results)

    if not existing_results_df.empty:
        combined_df = pd.concat([existing_results_df, session_df], ignore_index=True)
    else:
        combined_df = session_df

    combined_df.to_parquet(output_path, index=False)
    return len(combined_df)


def _advance():
    """
    Move to the next case. When the batch is complete, save all results
    to parquet and display a summary including accuracy statistics.
    """
    state["current_index"] += 1

    if state["current_index"] < len(case_list):
        load_case(state["current_index"])

    else:
        # ---- Batch complete ----

        # Disable all interactive controls
        stage1_submit.disabled          = True
        stage1_textarea.disabled        = True
        stage2_submit.disabled          = True
        stage2_radio.disabled           = True
        stage2_revise_textarea.disabled = True

        total_saved = save_results()
        n_session   = len(session_results)
        n_correct   = sum(1 for r in session_results if r["llm_correct"])
        n_incorrect = n_session - n_correct

        with completion_output:
            clear_output()
            print("✅  Batch complete!\n")
            print(f"   This session  : {n_session} case(s) reviewed")
            print(
                f"   LLM correct   : {n_correct} "
                f"({100 * n_correct   / n_session:.1f}%)"
            )
            print(
                f"   LLM incorrect : {n_incorrect} "
                f"({100 * n_incorrect / n_session:.1f}%)"
            )
            print(f"   Total saved   : {total_saved} case(s)")
            print(f"   Output file   : {output_path.resolve()}\n")
            if remaining_after_batch > 0:
                print(
                    f"   ℹ️  {remaining_after_batch} case(s) remain. "
                    "Re-run Cell 2 to start the next batch."
                )
            else:
                print("   🎉 All cases have been reviewed!")


# =============================================================================
# SECTION 9: Event handlers
# =============================================================================

def on_stage1_submit(button):
    """
    Handle submission of the human parse in Stage 1.

    Steps:
      1. Validate JSON syntax.
      2. Validate against the address_components schema.
      3. Store the validated parse in state['human_parsed'].
      4. Compare with the LLM parse:
           - Match    → record as correct (llm_correct=True) and advance.
           - Mismatch → switch to Stage 2 (comparison view).
    """
    stage1_feedback.clear_output()
    raw = stage1_textarea.value.strip()

    # --- Step 1: JSON syntax ---
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as exc:
        with stage1_feedback:
            print(f"⚠️  JSON syntax error — {exc}")
            print("Please correct and click Submit again.")
        return

    # --- Step 2: Schema validation ---
    schema_errors = validate_address_components(parsed)
    if schema_errors:
        with stage1_feedback:
            print("⚠️  Schema validation failed:")
            for err in schema_errors:
                print(f"   • {err}")
            print("\nPlease fix the issue(s) and click Submit again.")
        return

    # --- Step 3: Store the validated parse ---
    state["human_parsed"] = parsed

    # --- Step 4: Compare with LLM ---
    row        = case_list[state["current_index"]]
    llm_parsed = parse_components(row["address_components"])

    if are_equivalent(parsed, llm_parsed):
        _record_result(llm_correct=True)
        _advance()
    else:
        show_stage2()


def on_stage2_radio_change(change):
    """
    Show the revise textarea when 'Revise my parse' is selected;
    hide it when 'Mark LLM as incorrect' is selected.
    """
    if change["new"] == "Revise my parse":
        stage2_revise_section.layout.display = ""
    else:
        stage2_revise_section.layout.display = "none"


def on_stage2_submit(button):
    """
    Handle submission in Stage 2 (comparison view).

    'Mark LLM as incorrect':
      Record the result (llm_correct=False) using the most recently
      validated human parse from state['human_parsed'], then advance.

    'Revise my parse':
      1. Validate the revised JSON syntax.
      2. Validate against schema.
      3. Update state['human_parsed'] and refresh the human parse display.
      4. Re-compare with the LLM:
           - Match    → record as correct (llm_correct=True) and advance.
           - Mismatch → show a message and remain in Stage 2 so the reviewer
                        can revise again or switch to 'Mark LLM as incorrect'.
    """
    stage2_feedback.clear_output()

    if stage2_radio.value == "Mark LLM as incorrect":
        _record_result(llm_correct=False)
        _advance()
        return

    # --- "Revise my parse" path ---

    raw = stage2_revise_textarea.value.strip()

    # Step 1: JSON syntax
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as exc:
        with stage2_feedback:
            print(f"⚠️  JSON syntax error — {exc}")
            print("Please correct and click Submit again.")
        return

    # Step 2: Schema validation
    schema_errors = validate_address_components(parsed)
    if schema_errors:
        with stage2_feedback:
            print("⚠️  Schema validation failed:")
            for err in schema_errors:
                print(f"   • {err}")
            print("\nPlease fix the issue(s) and click Submit again.")
        return

    # Step 3: Update human parse state and refresh the Stage 2 display.
    # The LLM parse display (stage2_llm_html) is left unchanged.
    state["human_parsed"]   = parsed
    stage2_human_html.value = format_components_html(parsed, "Your parse:")

    # Step 4: Re-compare with LLM
    row        = case_list[state["current_index"]]
    llm_parsed = parse_components(row["address_components"])

    if are_equivalent(parsed, llm_parsed):
        _record_result(llm_correct=True)
        _advance()
    else:
        with stage2_feedback:
            print("⚠️  Your revised parse still differs from the LLM output.")
            print(
                "You can revise again, or select "
                "'Mark LLM as incorrect' if you believe your parse is right."
            )


# =============================================================================
# SECTION 10: Wire up event handlers
# =============================================================================

stage1_submit.on_click(on_stage1_submit)
stage2_radio.observe(on_stage2_radio_change, names="value")
stage2_submit.on_click(on_stage2_submit)

# =============================================================================
# SECTION 11: Assemble and launch the UI
# =============================================================================

ui = widgets.VBox(
    [
        progress_label,
        widgets.HTML('<hr style="margin:4px 0;">'),
        info_html,
        widgets.HTML('<hr style="margin:4px 0;">'),
        stage1_container,     # shown at startup; hidden when Stage 2 is active
        stage2_container,     # hidden at startup; shown on parse mismatch
        completion_output,
    ],
    layout=widgets.Layout(max_width="800px", padding="12px"),
)

# Only launch the UI if there are cases to review.
if case_list:
    load_case(0)
    display(ui)

Loaded 500 row(s) from: /proj/characklab/projects/kieranf/CMBS/analysis/geocoding/geocoding_input/consistent_parsed_addresses_random_sample.parquet
Found existing results: 10 case(s) already reviewed.

490 unprocessed case(s) found.
This session: reviewing 10 case(s). 480 case(s) will remain after this batch.
